In [1]:
"""
Weight Sharing — Cross-Layer Weight Tying
==========================================
Ties weights between structurally identical layers within the same ResNet stage.
This is the vision analogue of embedding↔output weight tying in NLP transformers.

METHOD
======
  ResNet50 has 4 stages (layer1–4). Each stage contains multiple Bottleneck blocks.
  Every Bottleneck block has:
    - conv1: 1×1  (channel reduction)
    - conv2: 3×3  (spatial feature extraction)  ← TIED across blocks in same stage
    - conv3: 1×1  (channel expansion)

  Within a stage, all conv2 layers share the EXACT same weight tensor:
    W_stage_k  is ONE nn.Parameter shared by all blocks in stage k

  Initialisation: average of all conv2 weights in the stage (better than just first).
  BN layers, conv1, conv3, downsample projections remain independent per block.

WHY VALID FOR VISION
====================
  Within a ResNet stage all conv2 layers have the same shape (C_in=C_out=same dim).
  The feature maps processed at each depth within a stage are semantically similar
  (same resolution, same channel count) — making weight sharing a reasonable inductive
  bias, similar to how Conv layers already share weights spatially.

  This is fundamentally different from NLP tying (embedding ↔ output projection),
  but serves the same compression goal: one parameter set, used multiple times.

COMPRESSION
===========
  ResNet50 stages:
    layer1: 3 blocks  → 3× conv2 tensors → tied to 1  (saves 2× conv2 weight)
    layer2: 4 blocks  → 4× conv2 tensors → tied to 1  (saves 3× conv2 weight)
    layer3: 6 blocks  → 6× conv2 tensors → tied to 1  (saves 5× conv2 weight)
    layer4: 3 blocks  → 3× conv2 tensors → tied to 1  (saves 2× conv2 weight)

OUTPUT
======
  __1__cross_layer_weight_tying.json
"""


import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch : {torch.__version__}", flush=True)
print(f"[init] Device  : {DEVICE}", flush=True)

# ── Config ────────────────────────────────────────────────────────────────────
ORIGINAL_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
MODEL_ARCH            = "resnet50"
OUTPUT_JSON           = "__1__cross_layer_weight_tying.json"

NUM_CLASSES  = 10
BATCH_SIZE   = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS  = 0
PIN_MEMORY   = DEVICE.type == "cuda"

FINETUNE_EPOCHS = 5
FINETUNE_LR     = 1e-3

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] FineTuneEpochs: {FINETUNE_EPOCHS}  lr: {FINETUNE_LR}", flush=True)

[init] PyTorch : 2.12.0.dev20260324+cu128
[init] Device  : cuda
[init] FineTuneEpochs: 5  lr: 0.001


In [2]:
# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(arch="resnet50", num_classes=10):
    m = models.resnet50(weights=None) if arch == "resnet50" else models.resnet18(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m


def load_model(path, arch):
    print(f"[model] Loading {arch} from {path} ...", flush=True)
    m = build_model(arch, NUM_CLASSES)
    m.load_state_dict(torch.load(path, map_location="cpu"))
    return m.to(DEVICE).eval()


# ── Data ──────────────────────────────────────────────────────────────────────
def get_loaders():
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    train_ds = torchvision.datasets.CIFAR10("../data", train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10("../data", train=False, download=True, transform=test_tf)
    return (
        DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
        DataLoader(test_ds,  512,        shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
    )

In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)


def param_count(model):
    return sum(p.numel() for p in model.parameters())


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        preds.extend(model(x).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
    return {
        "accuracy" : float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall"   : float(recall_score(labels,    preds, average="macro", zero_division=0)),
        "f1"       : float(f1_score(labels,         preds, average="macro", zero_division=0)),
    }


def measure_inference(model: nn.Module, device, batch_size=128, runs=50) -> dict:
    """
    Returns a dict with per-image (single), batch, and throughput timings.
    Uses CUDA events on GPU for highest accuracy; falls back to perf_counter on CPU.
    """
    m = model.eval().to(device)
    use_cuda = device.type == "cuda"

    def _timed_run(inp: torch.Tensor, n: int) -> float:
        """Returns total elapsed seconds for n forward passes."""
        with torch.no_grad():
            # warm-up
            for _ in range(3):
                m(inp)
            if use_cuda:
                torch.cuda.synchronize()
                start_evt = torch.cuda.Event(enable_timing=True)
                end_evt   = torch.cuda.Event(enable_timing=True)
                start_evt.record()
                for _ in range(n):
                    m(inp)
                end_evt.record()
                torch.cuda.synchronize()
                return start_evt.elapsed_time(end_evt) / 1000.0  # ms → s
            else:
                t0 = time.perf_counter()
                for _ in range(n):
                    m(inp)
                return time.perf_counter() - t0

    # Single-image latency
    single = torch.randn(1, 3, 32, 32, device=device)
    single_s = _timed_run(single, runs)
    single_ms = single_s / runs * 1000

    # Batch latency
    batch = torch.randn(batch_size, 3, 32, 32, device=device)
    batch_s = _timed_run(batch, runs)
    batch_ms = batch_s / runs * 1000

    per_img_ms  = batch_ms / batch_size
    throughput  = batch_size * runs / batch_s   # images / second

    timing_method = (
        "CUDA events + torch.cuda.synchronize()" if use_cuda
        else "time.perf_counter() (CPU)"
    )

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms,  4),
        "per_img_gpu_ms"     : round(per_img_ms, 6),
        "throughput_imgs_sec": round(throughput, 2),
        "timing_method"      : timing_method,
    }


def compute_flops(model: nn.Module, device, input_size=(1, 3, 32, 32)) -> float:
    """
    Estimates FLOPs (as GFLOPs) by registering forward hooks on Conv2d and Linear layers.
    Counts multiply-accumulate operations × 2 (one mul + one add = 2 FLOPs).
    Returns GFLOPs (float).
    """
    m = model.eval().to(device)
    total_flops = [0]
    hooks = []

    def conv_hook(module, inp, out):
        # out: (N, C_out, H_out, W_out)
        N, C_out, H_out, W_out = out.shape
        C_in  = module.in_channels
        kH, kW = module.kernel_size if isinstance(module.kernel_size, tuple) else (module.kernel_size, module.kernel_size)
        groups = module.groups
        # MACs per output element = (C_in / groups) * kH * kW
        macs = N * C_out * H_out * W_out * (C_in // groups) * kH * kW
        total_flops[0] += 2 * macs

    def linear_hook(module, inp, out):
        # out: (N, C_out)
        in_f  = module.in_features
        out_f = module.out_features
        N     = inp[0].shape[0]
        total_flops[0] += 2 * N * in_f * out_f

    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            hooks.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            hooks.append(mod.register_forward_hook(linear_hook))

    dummy = torch.randn(*input_size, device=device)
    with torch.no_grad():
        m(dummy)

    for h in hooks:
        h.remove()

    return round(total_flops[0] / 1e9, 6)  # GFLOPs



In [4]:
# ── Cross-Layer Weight Tying ──────────────────────────────────────────────────
def apply_weight_tying(model: nn.Module) -> tuple:
    """
    For each ResNet stage (layer1–4):
      - Collect all conv2 (3×3) layers across blocks in that stage
      - Verify shapes match exactly
      - Replace all their weight parameters with a single shared nn.Parameter
        initialised as the average of all block conv2 weights
      - BN, conv1, conv3 remain untouched per block

    Returns (tied_model, tying_info_dict)
    """
    model = copy.deepcopy(model)
    tying_info = {}
    total_params_saved = 0

    for stage_name in ["layer1", "layer2", "layer3", "layer4"]:
        stage  = getattr(model, stage_name)
        blocks = list(stage.children())

        # Collect conv2 layers from each block
        conv2_layers = [
            b.conv2 for b in blocks
            if hasattr(b, "conv2") and isinstance(b.conv2, nn.Conv2d)
        ]

        if len(conv2_layers) < 2:
            print(f"  [tying] {stage_name}: only {len(conv2_layers)} block(s), skipping",
                  flush=True)
            continue

        # Verify all shapes match exactly (required for tying)
        ref_shape = conv2_layers[0].weight.shape
        if not all(c.weight.shape == ref_shape for c in conv2_layers):
            print(f"  [tying] {stage_name}: shape mismatch, skipping", flush=True)
            continue

        # Shared parameter = mean of all block conv2 weights (better init than first-only)
        avg_weight   = torch.stack([c.weight.data for c in conv2_layers]).mean(0)
        shared_param = nn.Parameter(avg_weight)

        # Bind every conv2 in this stage to the same parameter object
        for c in conv2_layers:
            c.weight = shared_param

        n_blocks     = len(conv2_layers)
        params_each  = ref_shape.numel()
        params_saved = params_each * (n_blocks - 1)
        total_params_saved += params_saved

        tying_info[stage_name] = {
            "num_blocks_tied" : n_blocks,
            "weight_shape"    : list(ref_shape),
            "params_per_copy" : params_each,
            "params_saved"    : params_saved,
            "reduction_pct"   : round((1 - 1 / n_blocks) * 100, 2),
        }
        print(f"  [tying] {stage_name}: tied {n_blocks} blocks × "
              f"conv2{tuple(ref_shape)}  saved={params_saved:,} params", flush=True)

    tying_info["total_params_saved"] = total_params_saved
    print(f"  [tying] Total params saved: {total_params_saved:,}", flush=True)
    return model, tying_info


# ── Fine-tuning ───────────────────────────────────────────────────────────────
def finetune(model, train_loader, epochs, lr):
    if epochs == 0:
        return []
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    accs = []
    print(f"\n  Fine-tuning {epochs} epochs @ lr={lr} ...", flush=True)
    for epoch in range(epochs):
        correct = total = 0
        for inputs, targets in train_loader:
            inputs  = inputs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                loss = F.cross_entropy(model(inputs), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            with torch.no_grad():
                preds    = model(inputs).argmax(1)
                correct += (preds == targets).sum().item()
                total   += targets.size(0)
        scheduler.step()
        acc = correct / total
        accs.append(round(acc, 6))
        print(f"    Epoch {epoch+1}/{epochs}  train_acc={acc:.4f}", flush=True)
    return accs

In [5]:
# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Weight Sharing — Cross-Layer Weight Tying")
    print(f"  Model   : {MODEL_ARCH}  |  Device: {DEVICE}")
    print(f"  FineTune: {FINETUNE_EPOCHS} epochs @ lr={FINETUNE_LR}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline accuracy : {baseline_acc:.4f}\n", flush=True)

    original_model   = load_model(ORIGINAL_MODEL_PATH, MODEL_ARCH)
    original_size_mb = model_size_mb(original_model)
    original_params  = param_count(original_model)
    print(f"  Original size     : {original_size_mb:.2f} MB", flush=True)
    print(f"  Original params   : {original_params:,}\n", flush=True)

    train_loader, test_loader = get_loaders()

    # ── Apply tying ───────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    print("  Applying cross-layer weight tying ...", flush=True)
    tied_model, tying_info = apply_weight_tying(original_model)
    tied_model    = tied_model.to(DEVICE)
    compress_time = time.perf_counter() - t0
    print(f"  Done in {compress_time:.2f}s", flush=True)

    compressed_params = param_count(tied_model)
    param_reduction   = 1.0 - compressed_params / original_params
    print(f"  Compressed params : {compressed_params:,}  "
          f"({param_reduction*100:.1f}% reduction)\n", flush=True)

    # Pre-finetune evaluation
    pre_ft = evaluate(tied_model, test_loader)
    print(f"  Pre-FT accuracy   : {pre_ft['accuracy']:.4f}", flush=True)

    # Fine-tuning
    ft_accs = finetune(tied_model, train_loader, FINETUNE_EPOCHS, FINETUNE_LR)

    # Final evaluation
    metrics  = evaluate(tied_model, test_loader)
    infer    = measure_inference(tied_model, DEVICE)
    flops_g  = compute_flops(tied_model, DEVICE)
    size_mb  = model_size_mb(tied_model)
    acc_drop = baseline_acc - metrics["accuracy"]

    print(f"\n  ✓ Results:")
    print(f"    Accuracy         : {metrics['accuracy']:.4f}  (drop={acc_drop:+.4f})")
    print(f"    Size             : {size_mb:.2f} MB  (original={original_size_mb:.2f} MB)")
    print(f"    Compression ratio: {original_size_mb/size_mb:.2f}×")
    print(f"    Param reduction  : {param_reduction*100:.1f}%")
    print(f"    FLOPs            : {flops_g:.4f} G")
    print(f"    Inference (single): {infer['single_img_gpu_ms']:.2f} ms")
    print(f"    Inference (batch) : {infer['batch128_gpu_ms']:.2f} ms  "
          f"({infer['throughput_imgs_sec']:.0f} img/s)")

    model_save_path = os.path.join(
        os.path.dirname(os.path.abspath(OUTPUT_JSON)),
        "__1__cross_layer_weight_tying.pth"
    )
    torch.save(tied_model.state_dict(), model_save_path)
    print(f"  Saved model → {model_save_path}", flush=True)

    report = {
        "experiment"             : "weight_sharing_tying",
        "method"                 : "weight_sharing",
        "variant"                : "CrossLayerWeightTying",
        "original_arch"          : MODEL_ARCH,
        "dataset"                : "CIFAR-10",
        "train_device"           : str(DEVICE),
        # hyperparameters
        "finetune_epochs"        : FINETUNE_EPOCHS,
        "finetune_lr"            : FINETUNE_LR,
        # method description
        "method_description"     : (
            "Cross-layer weight tying: within each ResNet stage (layer1–4), all 3×3 conv2 "
            "layers across bottleneck blocks share one nn.Parameter (average-initialised). "
            "BN layers, conv1 (1×1), conv3 (1×1), and downsample projections remain "
            "independent per block. Vision analogue of NLP embedding-output weight tying."
        ),
        "tying_info"             : tying_info,
        # timing
        "compression_time_s"     : round(compress_time, 2),
        # baseline
        "baseline"               : baseline,
        "original_size_mb"       : round(original_size_mb, 4),
        "original_params"        : original_params,
        # performance
        "pre_finetune_accuracy"  : round(pre_ft["accuracy"], 6),
        "accuracy"               : round(metrics["accuracy"],  6),
        "precision"              : round(metrics["precision"], 6),
        "recall"                 : round(metrics["recall"],    6),
        "f1"                     : round(metrics["f1"],        6),
        "accuracy_drop"          : round(acc_drop, 6),
        "finetune_epoch_accs"    : ft_accs,
        # size & efficiency
        "params"                 : compressed_params,
        "original_params"        : original_params,
        "param_reduction_pct"    : round(param_reduction * 100, 2),
        "size_mb"                : round(size_mb, 4),
        "original_size_mb"       : round(original_size_mb, 4),
        "compression_ratio"      : round(original_size_mb / size_mb, 4) if size_mb else None,
        "flops_G"                : flops_g,
        "inference_ms"           : infer,
        "saved_model_path"       : model_save_path,
        "status"                 : "ok",
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved → {OUTPUT_JSON}")


if __name__ == "__main__":
    main()


  Weight Sharing — Cross-Layer Weight Tying
  Model   : resnet50  |  Device: cuda
  FineTune: 5 epochs @ lr=0.001

  Baseline accuracy : 0.9320

[model] Loading resnet50 from ../__2__baseline_resnet50_cifar10.pth ...
  Original size     : 90.03 MB
  Original params   : 23,520,842

  Applying cross-layer weight tying ...
  [tying] layer1: tied 3 blocks × conv2(64, 64, 3, 3)  saved=73,728 params
  [tying] layer2: tied 4 blocks × conv2(128, 128, 3, 3)  saved=442,368 params
  [tying] layer3: tied 6 blocks × conv2(256, 256, 3, 3)  saved=2,949,120 params
  [tying] layer4: tied 3 blocks × conv2(512, 512, 3, 3)  saved=4,718,592 params
  [tying] Total params saved: 8,183,808
  Done in 0.04s
  Compressed params : 15,337,034  (34.8% reduction)

  Pre-FT accuracy   : 0.1136

  Fine-tuning 5 epochs @ lr=0.001 ...
    Epoch 1/5  train_acc=0.8337
    Epoch 2/5  train_acc=0.8891
    Epoch 3/5  train_acc=0.9083
    Epoch 4/5  train_acc=0.9232
    Epoch 5/5  train_acc=0.9339

  ✓ Results:
    Accuracy 